In [1]:
import pandas as pd
import numpy as np
import glob, os, shutil
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize as MplNormalize


def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

def formatAxis(img):
    return np.transpose(img, (0, 2, 1))

def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

def showTile(img=None, mask=None, save=None):
    if img is None and mask is None:
        return print("Erro: Forneça pelo menos 'img' ou 'mask'.")

    ref_vol = img if img is not None else mask
    mid_x = ref_vol.shape[0] // 2
    mid_y = ref_vol.shape[1] // 2
    mid_z = ref_vol.shape[2] // 2

    def get_slices(vol):
        if vol is None:
            return None
        
        s_x = np.array(vol[mid_x, :, :]) # Plano YZ
        s_y = np.array(vol[:, mid_y, :]) # Plano XZ
        s_z = np.array(vol[:, :, mid_z]) # Plano XY
        return [s_x, np.rot90(s_z, -1), s_y]

    img_slices  = get_slices(img)
    mask_slices = get_slices(mask)
    cmap_mask_only    = ListedColormap(['black', 'red', 'green', 'blue'])
    cmap_mask_overlay = ListedColormap([(0, 0, 0, 0), 'red', 'green', 'blue'])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles    = [f'Slice X={mid_x}', f'Slice Y={mid_y}', f'Slice Z={mid_z}']

    for i, ax in enumerate(axes):
        if img is not None:
            ax.imshow(img_slices[i], cmap='gray')
            
        if mask is not None:
            if img is not None:
                ax.imshow(mask_slices[i], cmap=cmap_mask_overlay, vmin=0, vmax=3, alpha=0.6)
            else:
                ax.imshow(mask_slices[i], cmap=cmap_mask_only, vmin=0, vmax=3)
        
        ax.set_title(titles[i])

    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)
        return plt.close(fig)

    plt.show()


def show3DCube(ax, volume, label, x_ratio=0.1, y_ratio=0.9, z_ratio=0.9, stride=1):
    nx, ny, nz = volume.shape
    pos_x, pos_y, pos_z = int(nx * x_ratio), int(ny * y_ratio), int(nz * z_ratio)

    cmap = plt.cm.gray
    norm = MplNormalize(vmin=volume.min(), vmax=volume.max())

    def plot_plane(axis_to_fix, fixed_pos):
        if axis_to_fix == 'y':    # Plano XZ
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        elif axis_to_fix == 'x':  # Plano YZ
            ranges_dim1 = [(0, pos_y + 1), (pos_y, ny)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        else:                     # Plano XY (z)
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_y + 1), (pos_y, ny)]

        for start1, end1 in ranges_dim1:
            for start2, end2 in ranges_dim2:
                arr1, arr2 = np.arange(start1, end1), np.arange(start2, end2)

                if axis_to_fix == 'y':
                    X, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    Y = np.full_like(X, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[start1:end1, fixed_pos, start2:end2]
                elif axis_to_fix == 'x':
                    Y, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    X = np.full_like(Y, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[fixed_pos, start1:end1, start2:end2]
                else:
                    X, Y = np.meshgrid(arr1, arr2, indexing='ij')
                    Z_plot = np.full_like(X, nz - fixed_pos)
                    data = volume[start1:end1, start2:end2, fixed_pos]

                ax.plot_surface(X, Y, Z_plot, facecolors=cmap(norm(data)), shade=False, antialiased=False, linewidth=0, rstride=stride, cstride=stride)

    plot_plane('y', pos_y) # Parede XZ
    plot_plane('x', pos_x) # Parede YZ
    plot_plane('z', pos_z) # Chão XY

    ax.set_xlim(0, nx)
    ax.set_ylim(0, ny)
    ax.set_zlim(0, nz)
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()
    ax.set_title(label, fontsize=14, fontweight='bold', loc='left')
    ax.view_init(elev=20, azim=-45)


def showSteps(steps, save=None):
    fig = plt.figure(figsize=(18, 12))
    for i, (volume, label) in enumerate(steps):
        ax = fig.add_subplot(2, 3, i + 1, projection='3d')
        show3DCube(ax, volume, label)
        
    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)

    plt.show()

In [2]:
import numpy as np
from tqdm import tqdm
import scipy.ndimage as ndimage
import os, json


class SyntheticGenerator:
    def __init__(self, shape=(128, 128, 128)):
        # ── Image Format ─────────────────────────────────────────────
        self.margin = 64                  # Buffer para absorver dobras extremas nas bordas com segurança
        self.finalShape = shape           # (nx, ny, nz) final output volume size

        # ── Refletividade (Estratigrafia) ────────────────────────────
        self.layerRange = (100, 230)      # Qtd de camadas. ↑ Imagem cheia de linhas finas. ↓ Blocos grossos e lisos.
        self.layerThickness = (1, 2)      # Espessura. ↑ Camadas mais grossas. ↓ Camadas bem fininhas.

        # ── Dobramentos (Folding) ────────────────────────────────────
        self.foldCount = (15, 30)         # Qtd de dobras. ↑ Imagem muito ondulada. ↓ Terreno plano.
        self.foldSigma = (8, 44)          # Largura da dobra. ↑ Dobras largas e suaves. ↓ Dobras curtas e apertadas.
        self.foldAmplitude = (-17, 17)    # Altura da dobra. ↑ Picos e vales extremos. ↓ Dobras rasas.
        self.foldDamping   = 1.5          # Perda de força. ↑ A dobra some rápido no fundo. ↓ A dobra desce até a base.
        self.foldBaseShift = (-1.6, 1.6)  # Posição Z. ↑/↓ Sobe ou desce o desenho inteiro na imagem.

        # ── Cisalhamento / Inclinação (Shearing) ─────────────────────
        self.shearOffset   = (-2.8, 2.8)  # Deslocamento lateral. ↑/↓ Empurra todo o bloco para o lado.
        self.shearGradient = (-0.1, 0.1)  # Inclinação (Mergulho). ↑ Camadas ficam na diagonal. ↓ Ficam na horizontal.

        # ── Falhas (Faulting) ────────────────────────────────────────
        self.faultCount = (4, 7)          # Qtd de falhas. ↑ Imagem toda fraturada. ↓ Imagem mais inteira.
        self.faultThrow = (0, 22)         # Tamanho do degrau. ↑ Desencontro gigante nas linhas. ↓ Quebra quase invisível.
        self.faultDipAngle = (20, 75)     # Ângulo. ↑ Falha quase em pé (vertical). ↓ Falha deitada.
        
        self.faultRoughness  = 3.3        # Textura do corte. ↑ Corte tremido/áspero. ↓ Corte liso como navalha.
        self.faultRoughSigma = 4.5        # Tamanho da tremedeira. ↑ Ondas grandes na falha. ↓ Ondinhas curtas.
        self.faultDecaySigma = (33, 83)   # Arrasto. ↑ A linha entorta muito antes de quebrar. ↓ Quebra seca.

        self.faultZoneWidth  = 1.2        # Espessura do rótulo. ↑ A máscara da falha fica grossa. ↓ Fica fina.
        self.faultThreshold  = 0.8        # Filtro de rótulo. ↑ Marca só falha grande. ↓ Marca qualquer rachadurazinha.

        self.faultCurveProb  = 0.30       # Chance de curvar. ↑ Falha faz formato de colher (lístrica). ↓ Falha reta.
        self.faultCurveMax   = 6.7        # Força da curva. ↑ Curva muito fechada. ↓ Curva leve.

        # ── Assinatura Sísmica (Wavelet) ─────────────────────────────
        self.waveletFreq = (81, 117)      # Resolução. ↑ Imagem super nítida. ↓ Imagem borrada e grossa.
        self.waveletDuration = 0.08       # "Eco" do sinal. ↑ O traço borra verticalmente. ↓ Sinal limpo e curto.
        self.waveletDt = 0.002            # Amostragem. ↑ Imagem pode ficar pixelada/serrilhada. ↓ Imagem contínua.

        # ── Ruído Final (Noise) ──────────────────────────────────────
        self.noiseLevel = (0.00, 0.10)    # Chuvisco. ↑ Imagem cheia de ruído (ruim). ↓ Imagem limpa (perfeita).

        self.nx = self.finalShape[0] + 2 * self.margin
        self.ny = self.finalShape[1] + 2 * self.margin
        self.nz = self.finalShape[2] + 2 * self.margin
        self.shape = (self.nx, self.ny, self.nz)

    def get(self):
        data = self.genReflectivity()
        data = self.applyFolding(data)
        data = self.applyShearing(data)
        data, mask = self.applyFaulting(data)
        image = self.applyWavelet(data)
        image = self.applyNoise(image)

        image = self.crop(image)
        mask  = self.crop(mask)
        image = (image - np.mean(image)) / (np.std(image) + 1e-8)
        return image.astype(np.float32), mask.astype(np.uint8)

    def set(self, options):
        for k, v in options.items():
            setattr(self, k, v)

    def _generate_single(self, args):
        import numpy as np
        import os
        
        i, imgDir, mskDir, seed = args
        np.random.seed(seed)
        image, mask = self.get()
        image, mask = np.transpose(image, (0, 2, 1)), np.transpose(mask, (0, 2, 1))
        
        np.save(os.path.join(imgDir, f"img_{i:04d}.npy"), image)
        np.save(os.path.join(mskDir, f"img_{i:04d}.npy"), mask)

    def dataset(self, n=200, outputDir="output", n_jobs=None):
        from Utils.index import setFolder
        import concurrent.futures
        import multiprocessing
        import os
        from tqdm import tqdm
        
        imgDir = os.path.join(outputDir, "images")
        mskDir = os.path.join(outputDir, "masks")
        setFolder(imgDir)
        setFolder(mskDir)

        if n_jobs is None:
            n_jobs = multiprocessing.cpu_count()
            
        base_seed = np.random.randint(0, 1000000)
        tasks = [(i, imgDir, mskDir, base_seed + i) for i in range(n)]
        
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
            list(tqdm(executor.map(self._generate_single, tasks), total=n, desc="Generating dataset"))

    def genReflectivity(self):
        """Create 1D layered reflectivity tiled across the volume."""
        r1d = np.zeros(self.nz)
        nLayers = np.random.randint(*self.layerRange)

        for _ in range(nLayers):
            pos = np.random.randint(0, self.nz)
            thickness = np.random.randint(*self.layerThickness)
            r1d[pos : pos + thickness] = np.random.uniform(-1, 1)

        return np.tile(r1d, (self.nx, self.ny, 1))
        
    def applyFolding(self, reflectivity):
        """Deform layers with rotated anisotropic Gaussian folds."""
        x = np.arange(self.nx)
        y = np.arange(self.ny)
        xx, yy = np.meshgrid(x, y, indexing="ij")

        a0 = np.random.uniform(*self.foldBaseShift)
        nGaussians = np.random.randint(*self.foldCount)
        shift2d    = np.zeros((self.nx, self.ny))

        for _ in range(nGaussians):
            x0 = np.random.uniform(-self.nx * 0.3, self.nx * 1.3)
            y0 = np.random.uniform(-self.ny * 0.3, self.ny * 1.3)
            sigmaX = np.random.uniform(*self.foldSigma)
            sigmaY = np.random.uniform(*self.foldSigma)
            theta  = np.random.uniform(0, np.pi)
            amp = np.random.uniform(*self.foldAmplitude)

            dx = xx - x0
            dy = yy - y0
            cosT, sinT = np.cos(theta), np.sin(theta)
            u = cosT * dx + sinT * dy
            v = -sinT * dx + cosT * dy
            shift2d += amp * np.exp(-(u**2 / (2 * sigmaX**2) + v**2 / (2 * sigmaY**2)))

        zGrid = np.arange(self.nz)
        damping = self.foldDamping * zGrid / (self.nz - 1)
        s1 = a0 + shift2d[:, :, np.newaxis] * damping

        ix, iy, iz = np.indices(self.shape)
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s1], order=3, mode="nearest")

    def applyShearing(self, reflectivity):
        """Apply linear shear (dip/tilt) along X and Y axes."""
        e0 = np.random.uniform(*self.shearOffset)
        f  = np.random.uniform(*self.shearGradient)
        g  = np.random.uniform(*self.shearGradient)

        ix, iy, iz = np.indices(self.shape)
        s2 = e0 + f * ix + g * iy
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s2], order=3, mode="nearest")

    def applyFaulting(self, reflectivity):
        """Inject faults with displacement and produce binary mask."""
        masks = np.zeros(self.shape, dtype=np.uint8)
        model = np.copy(reflectivity)

        numFaults  = np.random.randint(*self.faultCount)
        ix, iy, iz = np.indices(self.shape)

        for i in range(numFaults):
            p0 = np.random.uniform(0.15, 0.85, 3) * np.array(self.shape)

            dip_angle  = np.random.uniform(*self.faultDipAngle)
            dip_rad    = np.deg2rad(dip_angle)
            strike_rad = np.random.uniform(0, 2 * np.pi)
            nx = np.sin(dip_rad) * np.cos(strike_rad)
            ny = np.sin(dip_rad) * np.sin(strike_rad)
            nz = np.cos(dip_rad) * np.random.choice([-1.0, 1.0])
            normal = np.array([nx, ny, nz])

            strike = np.array([-normal[1], normal[0], 0.0])
            strikeNorm = np.linalg.norm(strike)
            strike = np.array([1.0, 0.0, 0.0]) if strikeNorm < 1e-6 else strike / strikeNorm
            dip = np.cross(normal, strike)
            dip /= np.linalg.norm(dip)

            dx = ix - p0[0]
            dy = iy - p0[1]
            dz = iz - p0[2]

            distStrike = strike[0] * dx + strike[1] * dy + strike[2] * dz
            distDip = dip[0] * dx + dip[1] * dy + dip[2] * dz
            bend    = 0.0
            
            if np.random.random() < self.faultCurveProb:
                max_dist = max(self.shape) / 1.5 
                intensidade_base = np.random.uniform(self.faultCurveMax * 0.5, self.faultCurveMax)
                direcao = np.random.choice([-1.0, 1.0])
                curve_intensity = intensidade_base * direcao
                bend = curve_intensity * ((distDip / max_dist) ** 2)

            noisePlane = ndimage.gaussian_filter(np.random.normal(0, 1, self.shape), sigma=self.faultRoughSigma) * self.faultRoughness
            distPlane  = normal[0] * dx + normal[1] * dy + normal[2] * dz + noisePlane - bend
            maxDisp  = np.random.uniform(*self.faultThrow)
            throwMap = self.computeThrowMap(distStrike, distDip, maxDisp)

            hw = distPlane > 0
            throw_hw = throwMap[hw]

            ixShifted = ix.astype(np.float32)
            iyShifted = iy.astype(np.float32)
            izShifted = iz.astype(np.float32)
            
            ixShifted[hw] += throw_hw * dip[0]
            iyShifted[hw] += throw_hw * dip[1]
            izShifted[hw] += throw_hw * dip[2]

            model = ndimage.map_coordinates(model, [ixShifted, iyShifted, izShifted], order=1, mode="nearest")
            masks = ndimage.map_coordinates(masks, [ixShifted, iyShifted, izShifted], order=0, mode="constant", cval=0)
            faultZone = (np.abs(distPlane) <= self.faultZoneWidth) & (np.abs(throwMap) > self.faultThreshold)
            masks[faultZone] = 1

        return model, masks

    def computeThrowMap(self, distStrike, distDip, maxDisp):
        """Compute displacement map for a single fault (gaussian or linear decay)."""
        if np.random.random() < 0.5:
            sigmaPlane = np.random.uniform(*self.faultDecaySigma)
            return maxDisp * np.exp(-(distStrike**2 + distDip**2) / (2 * sigmaPlane**2))

        planeExtent = np.sqrt(self.nx**2 + self.ny**2 + self.nz**2)
        direction   = np.random.choice([-1, 1])
        return maxDisp * np.clip(0.5 + direction * distDip / planeExtent, 0, 1)

    def applyWavelet(self, model):
        """Convolve with a Ricker wavelet along the Z axis."""
        f = np.random.uniform(*self.waveletFreq)
        t = np.arange(-self.waveletDuration, self.waveletDuration, self.waveletDt)
        wavelet = (1 - 2 * (np.pi * f * t) ** 2) * np.exp(-((np.pi * f * t) ** 2))
        return ndimage.convolve1d(model, wavelet, axis=2)

    def applyNoise(self, image):
        """Add band-limited Gaussian noise scaled to signal amplitude."""
        scale = np.random.uniform(*self.noiseLevel) * np.std(image)
        noise = np.random.normal(0.0, 1.0, image.shape)
        noise = ndimage.gaussian_filter(noise, sigma=(1.0, 1.0, 0.5))
        noise *= (scale / (np.std(noise) + 1e-8))
        
        image = (image + noise)
        image = ndimage.gaussian_filter(image, sigma=(0.5, 0.5, 0))
        return image

    def crop(self, volume):
        """Removes the safety margin to extract the final shape volume."""
        x0, x1 = self.margin, self.nx - self.margin
        y0, y1 = self.margin, self.ny - self.margin
        z0, z1 = self.margin, self.nz - self.margin
        return volume[x0:x1, y0:y1, z0:z1]

    def getMetrics(self):
        return {
            "shape": self.shape,
            "margin": self.margin,
            "layerRange": self.layerRange,
            "layerThickness": self.layerThickness,
            "foldCount": self.foldCount,
            "foldSigma": self.foldSigma,
            "foldAmplitude": self.foldAmplitude,
            "foldDamping": self.foldDamping,
            "foldBaseShift": self.foldBaseShift,
            "shearOffset": self.shearOffset,
            "shearGradient": self.shearGradient,
            "faultCount": self.faultCount,
            "faultThrow": self.faultThrow,
            "faultDipAngle": self.faultDipAngle,
            "faultRoughness": self.faultRoughness,
            "faultRoughSigma": self.faultRoughSigma,
            "faultDecaySigma": self.faultDecaySigma,
            "faultZoneWidth": self.faultZoneWidth,
            "faultThreshold": self.faultThreshold,
            "faultCurveProb": self.faultCurveProb,
            "faultCurveMax": self.faultCurveMax,
            "waveletFreq": self.waveletFreq,
            "waveletDuration": self.waveletDuration,
            "waveletDt": self.waveletDt,
            "noiseLevel": self.noiseLevel
        }
    
    def print(self):
        print(json.dumps(self.getMetrics(), indent=4))


gen = SyntheticGenerator()

In [3]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import sys, gc
sys.path.append("..")
from Network.index import ModelNetwork

In [4]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


In [5]:
baseDir = 'model/'
if not os.path.exists(baseDir):
    print(f"Error: Model dir not found at {baseDir}")

In [6]:
with open(f'{baseDir}/info.json', 'r', encoding='utf-8') as f:
    modelInfo = json.load(f)

modelOptions = modelInfo.get('model', {})
print("Model Options:")
print(json.dumps(modelOptions, indent=4))

network   = ModelNetwork(**modelOptions)
modelData = torch.load(f'{baseDir}/model.pth')

network.model.load_state_dict(modelData['model'])
network.model.eval()

Model Options:
{
    "network": "resaceunet",
    "img_size": [
        128,
        128,
        128
    ],
    "classes": 1,
    "channels": 1,
    "dropout": 0.1,
    "num_filters": 16,
    "lr": 0.001
}


ResACE_Unet(
  (enc1): ResACEBlock3D(
    (conv1): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (bn1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (bn2): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (dropout): Dropout3d(p=0.1, inplace=False)
    (relu): ReLU(inplace=True)
    (skip): Sequential(
      (0): Conv3d(1, 16, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (1): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (attention): EfficientChannelAttention3D(
      (avg_pool): AdaptiveAvgPool3d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=16, out_features=4, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=4, out_features=16, bias=False)
        (3): Sigmoid()
      )
    )
  )
  (poo

In [7]:
class PredictDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row  = self.df.iloc[index]
        img  = np.load(row.img_path).astype(np.float32)
        mask = np.load(row.mask_path).astype(np.float32)

        img_tensor  = torch.tensor(img,  dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        return (img_tensor, mask_tensor)

In [8]:
def computeIoU(network, loader):
    network.model.eval()
    network.iou.reset()
    
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Computing IoU"):
            imgs, masks = imgs.to(network.device), masks.to(network.device)
            logits = network.model(imgs)
            
            if network.multiclass:
                preds  = torch.argmax(logits, dim=1)
                target = masks.squeeze(1) if masks.dim() == 5 else masks
                network.iou.update(preds, target)
            else:
                preds = (torch.sigmoid(logits) > 0.5)
                network.iou.update(preds, masks.int())
                
    iouValue = network.iou.compute().item()
    return iouValue

In [9]:
def plotPrediction(index, dataset, network, alpha=0.5, savePath=None):
    img_tensor, mask_tensor = dataset[index]
    network.model.eval()

    with torch.no_grad():
        img_batch = img_tensor.unsqueeze(0).to(network.device)
        logits    = network.model(img_batch)
        pred = torch.argmax(logits, dim=1).squeeze() if network.multiclass else (torch.sigmoid(logits) > 0.5).squeeze().int()

    img_np  = img_tensor.squeeze().cpu().numpy()
    mask_np = mask_tensor.squeeze().cpu().numpy()
    pred_np = pred.cpu().numpy()

    imgNorm = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    imgRgb  = np.stack((imgNorm, imgNorm, imgNorm), axis=-1)
    overlay = imgRgb.copy()
    is_gt   = (mask_np > 0)
    is_pred = (pred_np > 0)
    inters  = (is_gt & is_pred)   # Acertos / Interseção

    overlay[is_gt]   = [0, 0, 255]  # Azul para o Ground Truth
    overlay[is_pred] = [255, 0, 0]  # Vermelho para a Predição
    overlay[inters]  = [0, 255, 0]  # verde onde Predição e GT convergem

    result = (imgRgb * (1 - alpha) + overlay * alpha).astype(np.uint8)
    
    if savePath:
        showTile(result, mask=None, save=savePath)
    else:
        showTile(result, mask=None)

In [ ]:
import random, string, shutil, re, os, glob, json, torch
import pandas as pd
from torch.utils.data import DataLoader


def get_random_config():
    layer_min = random.randint(10, 110)
    layer_max = random.randint(111, 450)     # Permite de blocos massivos a linhas ultra-finas
    thick_min = random.randint(1, 3)
    thick_max = random.randint(3, 10)       # Camadas bem mais grossas se necessário
    
    # 2. Dobramentos (Folds)
    # Original era (5~10, 21~30) e amplitude (- 5~10, 10~20)
    fold_cnt_min = random.randint(0, 12)    # 0 permite terrenos totalmente planos
    fold_cnt_max = random.randint(13, 50)   # Até 50 dobras para caos completo
    fold_sig_min = random.randint(3, 22)    # Dobras bem mais curtas/fechadas
    fold_sig_max = random.randint(23, 80)   # Dobras super largas/regionais
    fold_amp_min = -random.randint(0, 35)   # De nenhuma dobra a picos massivos
    fold_amp_max = random.randint(0, 45)
    
    # 3. Falhas (Faults) - Modificado para ser dinâmico também!
    # Original do faultCount era fixo em (4, 7). Agora varia de 0 a 14 falhas.
    fault_cnt_min = random.randint(0, 3)    
    fault_cnt_max = random.randint(4, 14)   
    fault_thr_min = random.randint(0, 8)    
    fault_thr_max = random.randint(9, 45)   # Degraus gigantescos nas falhas
    
    # Ângulo de mergulho: Original era (40~50, 75~85)
    # Agora permite falhas quase deitadas (bbaixas/thrust) até verticais puras
    dip_min = random.randint(10, 55)
    dip_max = random.randint(56, 89)

    # Wavelet (Resolução Sísmica)
    # Original freq era (40~90, 91~150)
    wave_freq_min = random.randint(15, 100)  # Frequências baixas geram imagens bem borradas
    wave_freq_max = random.randint(101, 220) # Frequências altas geram nitidez extrema

    return {
        'layerRange': (layer_min, layer_max),
        'layerThickness': (thick_min, thick_max),
        
        'foldCount': (fold_cnt_min, fold_cnt_max),
        'foldSigma': (fold_sig_min, fold_sig_max),
        'foldAmplitude': (fold_amp_min, fold_amp_max),
        'foldDamping': random.uniform(0.0, 5.0),            # De sem amortecimento a sumir instantaneamente
        'foldBaseShift': (-random.uniform(0.0, 4.0), random.uniform(0.0, 4.0)),
        
        'shearOffset': (-random.uniform(0.0, 8.0), random.uniform(0.0, 8.0)),
        'shearGradient': (-random.uniform(0.0, 0.4), random.uniform(0.0, 0.4)), # Permite inclinações severas
        
        'faultCount': (fault_cnt_min, fault_cnt_max),
        'faultThrow': (fault_thr_min, fault_thr_max),
        'faultDipAngle': (dip_min, dip_max),
        
        'faultRoughness': random.uniform(0.0, 5.0),          # De corte perfeito de navalha (0) a super serrilhado
        'faultRoughSigma': random.uniform(0.5, 12.0),
        'faultDecaySigma': (random.randint(5, 60), random.randint(61, 150)),
        
        'faultZoneWidth': random.uniform(0.5, 3.5),          # Máscaras de falha bem finas ou bem espessas
        'faultThreshold': random.uniform(0.1, 1.5),
        
        'faultCurveProb': random.uniform(0.0, 0.7),          # Até 70% de chance de falhas lístricas (curvas)
        'faultCurveMax': random.uniform(0.0, 9.0),
        
        'waveletFreq': (wave_freq_min, wave_freq_max),
        'waveletDuration': random.uniform(0.02, 0.20),
        'waveletDt': random.uniform(0.0001, 0.015),
        
        'noiseLevel': (0.0, random.uniform(0.0, 0.55))       # Permite testar a resiliência do modelo com dados bem ruidosos
    }

def run_automation_pipeline(num_variations=3, batch_size=5):
    base_output = 'synthetic'
    os.makedirs(base_output, exist_ok=True)
    
    # Determine starting index by finding the max variation_N in base_output
    existing_dirs = [d for d in os.listdir(base_output) if os.path.isdir(os.path.join(base_output, d)) and d.startswith('variation_')]
    start_idx = 0
    if existing_dirs:
        indices = [int(d.split('_')[1]) for d in existing_dirs if d.split('_')[1].isdigit()]
        if indices:
            start_idx = max(indices) + 1
            
    results_log = []
    
    for i in range(num_variations):
        config = get_random_config()
        
        config_id = f"variation_{start_idx + i}"
        config_dir = os.path.join(base_output, config_id)
        os.makedirs(config_dir, exist_ok=True)
        print(f"\n[{i+1}/{num_variations}] Generating Data for Config: {config_id}")
        
        try:
            gen = SyntheticGenerator()
            gen.set(config)
                
            gen.dataset(n=batch_size, outputDir=config_dir)
            imgPaths  = sorted(glob.glob(f'{config_dir}/images/*.npy'))
            maskPaths = sorted(glob.glob(f'{config_dir}/masks/*.npy'))
            
            df = pd.DataFrame({'img_path': imgPaths, 'mask_path': maskPaths})
            synthDataset = PredictDataset(df)
            synthLoader  = DataLoader(
                synthDataset, 
                batch_size=1,
                shuffle=False, 
                num_workers=0,
                pin_memory=False
            )
            
            # 4. Evaluation
            iouValue = computeIoU(network, synthLoader)
            print(f"IoU for config {config_id}: {iouValue:.4f}")
            
            # Logging & Visualizations
            plots_dir = os.path.join(config_dir, 'plots')
            os.makedirs(plots_dir, exist_ok=True)
            for j in range(len(synthDataset)):
                plotPath = os.path.join(plots_dir, f'plot_{j:04d}.png')
                plotPrediction(j, synthDataset, network, savePath=plotPath)
                
            metrics = gen.getMetrics()
            metrics['iou'] = iouValue
            
            with open(os.path.join(config_dir, 'config_log.json'), 'w') as f:
                json.dump(metrics, f, indent=4)
                
            results_log.append({
                'config_id': config_id,
                'iou': iouValue
            })
        except Exception as e:
            print(f"Error processing config {config_id}: {e}")
            results_log.append({'config_id': config_id, 'iou': -1, 'error': str(e)})


run_automation_pipeline(num_variations=50_000, batch_size=15)


[1/50000] Generating Data for Config: variation_178


Computing IoU: 100%|██████████| 15/15 [00:01<00:00,  8.47it/s]


IoU for config variation_178: 0.2343

[2/50000] Generating Data for Config: variation_179


Computing IoU: 100%|██████████| 15/15 [00:01<00:00,  9.30it/s]


IoU for config variation_179: 0.1081

[3/50000] Generating Data for Config: variation_180


...

Computing IoU: 100%|██████████| 15/15 [00:01<00:00,  9.45it/s]


IoU for config variation_201: 0.1856

[25/50000] Generating Data for Config: variation_202


Computing IoU: 100%|██████████| 15/15 [00:01<00:00,  9.42it/s]


IoU for config variation_202: 0.4959

[26/50000] Generating Data for Config: variation_203


Generating dataset:   0%|          | 0/15 [00:00<?, ?it/s]